# Lab 03: K-Nearest Neighbors - Classification
This lab continues from Lab 02. We turn distance computations into a complete k-NN classifier, study how `k` changes decision boundaries, and evaluate `k=1` and `k=5` on CIFAR-10.

## Setup
Run from the A1 directory.

In [ ]:
import torch
import matplotlib.pyplot as plt
import dlcv2026

plt.rcParams["figure.figsize"] = (10.0, 8.0)
plt.rcParams["font.size"] = 16

from knn import hello
hello()

## Confirm the Lab 02 distance function

In [ ]:
from knn import compute_distances_no_loops

x_train_small = torch.tensor([[0.0, 0.0], [3.0, 4.0]], dtype=torch.float64)
x_test_small = torch.tensor([[0.0, 4.0], [3.0, 0.0]], dtype=torch.float64)
expected = torch.tensor([[16.0, 9.0], [9.0, 16.0]], dtype=torch.float64)
actual = compute_distances_no_loops(x_train_small, x_test_small)
assert torch.allclose(actual, expected), f"Expected {expected}, got {actual}"
print("Distance function ready for Lab 03.")

## Predict labels
Implemented in `knn.py` as `predict_labels` (top-k nearest neighbors + majority vote, ties go to the smallest label).

In [ ]:
from knn import predict_labels

torch.manual_seed(0)
dists = torch.tensor([
    [0.3, 0.4, 0.1],
    [0.1, 0.5, 0.5],
    [0.4, 0.1, 0.2],
    [0.2, 0.2, 0.4],
    [0.5, 0.3, 0.3],
])
y_train = torch.tensor([0, 1, 0, 1, 2])
y_pred_expected = torch.tensor([1, 0, 0])
y_pred = predict_labels(dists, y_train, k=3)
correct = y_pred.tolist() == y_pred_expected.tolist()
print('Correct: ', correct)

In [ ]:
# Extra check: tie-breaking returns the smallest label
dists_tie = torch.tensor([[0.1], [0.2], [0.3], [0.4], [0.5]])
y_tie = torch.tensor([1, 2, 1, 2, 3])
print(predict_labels(dists_tie, y_tie, k=5))  # expected: tensor([1])

## KnnClassifier on toy 2D data
Training points are stars, test points are transparent circles; color = label.

In [ ]:
from knn import KnnClassifier

num_test = 10000
num_train = 20
num_classes = 5

torch.manual_seed(128)
x_train = torch.rand(num_train, 2)
y_train = torch.randint(num_classes, size=(num_train,))
x_test = torch.rand(num_test, 2)
classifier = KnnClassifier(x_train, y_train)

for k in [1, 3, 5]:
    y_test = classifier.predict(x_test, k=k)
    plt.gcf().set_size_inches(8, 8)
    class_colors = ['r', 'g', 'b', 'k', 'y']
    train_colors = [class_colors[c] for c in y_train]
    test_colors = [class_colors[c] for c in y_test]
    plt.scatter(x_test[:, 0], x_test[:, 1],
                color=test_colors, marker='o', s=32, alpha=0.05)
    plt.scatter(x_train[:, 0], x_train[:, 1],
                color=train_colors, marker='*', s=128.0)
    plt.title('Predictions for k = %d' % k, size=16)
    plt.show()

## CIFAR-10, k=1
Expected accuracy: about 27%.

In [ ]:
from knn import KnnClassifier

torch.manual_seed(0)
num_train = 5000
num_test = 500
x_train, y_train, x_test, y_test = dlcv2026.data.cifar10(num_train, num_test)

classifier = KnnClassifier(x_train, y_train)
classifier.check_accuracy(x_test, y_test, k=1)

## CIFAR-10, k=5
Expected a slightly higher accuracy than k=1.

In [ ]:
from knn import KnnClassifier

torch.manual_seed(0)
num_train = 5000
num_test = 500
x_train, y_train, x_test, y_test = dlcv2026.data.cifar10(num_train, num_test)

classifier = KnnClassifier(x_train, y_train)
classifier.check_accuracy(x_test, y_test, k=5)

## Wrap-up
**1. How did the decision boundary change as k increased from 1 to 5?**
With k=1 every training point owns a small region around itself, so boundaries are fragmented and jagged and a single noisy/outlier point creates its own island. As k grows, predictions are decided by a vote among several neighbors, so isolated islands vanish and the boundaries become smoother and regions larger and more regular.

**2. Which value performed better on the CIFAR-10 subset?**
(Fill in after running: k=1 got ____%, k=5 got ____%.) k=5 should be slightly better, because voting over several neighbors reduces the influence of individual noisy neighbors.

**3. Why is test-set accuracy not the right quantity for choosing k?**
If we pick k by looking at test accuracy, the test set is used to tune a hyperparameter, so it is no longer unseen data and the reported accuracy becomes an optimistic estimate of real performance. k should be chosen on a validation set (or by cross-validation on the training data), and the test set used only once at the end.